In [1]:
import pandas as pd

In [9]:
df = pd.read_csv(r"C:\Users\NW PC\Desktop\Python Projects\Python 2\customer_transactions.csv")

In [12]:
df['order_date'] = pd.to_datetime(df['order_date'])
print("Raw transactions shape:",df.shape)
print(df.head())
print("\n")

Raw transactions shape: (3766, 4)
  transaction_id customer_id order_date  order_value
0     TXN_001754   CUST_0230 2023-12-18        68.15
1     TXN_002754   CUST_0354 2023-12-22        35.56
2     TXN_003428   CUST_0455 2024-01-03        15.68
3     TXN_002427   CUST_0313 2024-01-06        39.15
4     TXN_000088   CUST_0012 2024-01-14        60.27




In [13]:
# Define our snapshot date
# RFM always need a fixed reference point in time "as of today, how recent/frequent/valuable is this customer?"
# We use the day after the last transaction in the dateset as a stand-in for today

In [15]:
snapshot_date = df['order_date'].max() + pd.Timedelta(days=1)
print("Snapshot(analysis)date:", snapshot_date.date(), "\n")

Snapshot(analysis)date: 2025-12-27 



In [16]:
# Calculate R, F, M per customer

In [20]:
rfm = df.groupby('customer_id').agg(last_purchase_date=('order_date','max'), frequency = ('transaction_id', 'count'), 
                                    monetary = ('order_value','sum')).reset_index()

In [21]:
# Calculate Recency (days between snapshot date and the last purchase.
# The smaller the number, the most recent, and the better

In [23]:
rfm['recency'] = (snapshot_date - rfm['last_purchase_date']).dt.days
print("raw RFM table(first 5 customers):")
print(rfm.head())
print("\n")

raw RFM table(first 5 customers):
  customer_id last_purchase_date  frequency  monetary  recency
0   CUST_0001         2025-12-16         24   2113.93       11
1   CUST_0002         2024-07-11          1     44.30      534
2   CUST_0003         2025-05-31          5    210.74      210
3   CUST_0004         2025-07-16          1     75.17      164
4   CUST_0005         2025-02-19          3    145.21      311




In [35]:
# Score each dimension from 1(worst) to 4(best) using pd.qcut
# For recency, since LOWER days since last purchase is BETTER, we reverse the numbers to be 4,3,2,1

In [39]:
rfm['r_score'] = pd.qcut(rfm['recency'], q=4, labels=[4,3,2,1]).astype(int)

In [36]:
# For frequency and monetary HIGHER is BETTER

In [40]:
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)

In [42]:
rfm['m_score'] = pd.qcut(rfm['monetary'], q=4, labels=[1,2,3,4]).astype(int)

In [44]:
rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

print("Scored RFM table(first 5 customers):")
print(rfm[['customer_id', 'recency', 'frequency', 'monetary', 'r_score', 'f_score', 'm_score', 'rfm_score']].head())
print("\n")

Scored RFM table(first 5 customers):
  customer_id  recency  frequency  monetary  r_score  f_score  m_score  \
0   CUST_0001       11         24   2113.93        4        4        4   
1   CUST_0002      534          1     44.30        1        1        1   
2   CUST_0003      210          5    210.74        2        2        2   
3   CUST_0004      164          1     75.17        2        1        1   
4   CUST_0005      311          3    145.21        1        2        2   

  rfm_score  
0       444  
1       111  
2       222  
3       211  
4       122  




In [51]:
def segment_customer(row):
    r,f,m = row['r_score'], row['f_score'], row['m_score']
    if r>= 4 and f>=4 and m>=4:
        return 'Champion'
    elif r>=3 and f>=3:
        return 'Loyal Customer'
    elif r<= 2 and f>=3:
        return 'At Risk'
    elif r>=3 and f<=2:
        return 'New'
    else:
        return'Lost'
    
rfm['segment'] = rfm.apply(segment_customer, axis =1)

In [ ]:
# Business Summary

In [54]:
segment_summary = rfm.groupby('segment').agg(
    customer_count=('customer_id','count'),
    avg_monetary=('monetary', 'mean'),
    total_monetary=('monetary', 'sum')
).sort_values('total_monetary', ascending = False)

print("Segment summary: ")
print(segment_summary.round(2))
print("\n")


Segment summary: 
                customer_count  avg_monetary  total_monetary
segment                                                     
Champion                    72       2286.07       164597.20
Loyal Customer             122        907.45       110709.19
At Risk                     56        634.79        35548.45
Lost                       194        139.56        27075.57
New                         56        175.29         9816.32




In [55]:
# Save the full customer level RFM table for marketing to act on

In [57]:
rfm.to_csv('customer_rfm_segments.csv', index=False)
print("Saved: customer_rfm_segments.csv")
print("\n")

Saved: customer_rfm_segments.csv




In [58]:
# Insight Summary

In [62]:
total_revenue=rfm['monetary'].sum()

champion_revenue_share = (segment_summary.loc[
    'Champion', 'total_monetary']/total_revenue * 100
                          if 'Champion' in segment_summary.index else 0
                         )
at_risk_count = (segment_summary.loc['At Risk', 'customer_count']
                     if 'At Risk' in segment_summary.index else 0
                    )
                     

In [63]:
print("=" * 50)
print("SUMMARY OF INSIGHTS")
print("=" * 50)
print(f"Total customers analyzed: {len(rfm)}")
print(f"Champions generate {champion_revenue_share:.1f}% of total revenue")
print(f"Customers flagged 'At Risk'(act fast):{at_risk_count}")

SUMMARY OF INSIGHTS
Total customers analyzed: 500
Champions generate 47.3% of total revenue
Customers flagged 'At Risk'(act fast):56
